# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


## Load Data

In [2]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260310.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [3]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=50, max_na_per_row=6, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11185 rows
Basic cleaning complete: 8576 rows remaining

Starting advanced column cleaning with 3097 columns

Advanced column cleaning complete: 3097 → 2086 columns (1011 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 98
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 6 NaN values...
Removed 3922 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4654, 2086)


In [4]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1895,total_consensus_pct_under,72,1.55,2024.0
1896,spread_consensus_pct_home,67,1.44,2024.0
1749,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,15,0.32,2023.0
1755,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,15,0.32,2023.0
1751,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,14,0.30,2023.0
1903,ml_consensus_opener_price_home,14,0.30,2025.0
1902,ml_consensus_opener_price_away,14,0.30,2025.0
1466,spread_betmgm_price_LAST_HOME_AWAY_5_MATCHES_D...,12,0.26,2021.0
1430,total_betmgm_price_under_LAST_HOME_AWAY_5_MATC...,12,0.26,2021.0
1427,total_betmgm_price_over_LAST_HOME_AWAY_5_MATCH...,12,0.26,2021.0


In [5]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [6]:
# df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train['TOTAL_OVER_UNDER_LINE']
df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train[BET365_LINE_COL]

In [7]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

## Train / Test

In [8]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.base import clone

In [9]:
def _advance_until_n_games(date_counts: pd.Series, start_pos: int, target_games: int):
    end_pos = start_pos
    n_games = 0

    while end_pos < len(date_counts) and n_games < target_games:
        n_games += int(date_counts.iloc[end_pos])
        end_pos += 1

    return end_pos, n_games


def make_walk_forward_last_n_seasons_splits(
    df: pd.DataFrame,
    date_col: str = "GAME_DATE",
    season_col: str = "SEASON_YEAR",
    train_seasons: int = 3,
    test_games: int = 30,
    step_games: int | None = None,
    min_train_games: int = 300,
    max_folds: int | None = None,
    random_state: int | None = 16,
    verbose: int = 1,
):
    if step_games is None:
        step_games = test_games

    temp = df[[date_col, season_col]].copy()
    temp["_pos"] = np.arange(len(temp))
    temp["_date"] = pd.to_datetime(temp[date_col], errors="coerce").dt.normalize()

    if temp["_date"].isna().any():
        raise ValueError(f"Invalid dates found in {date_col}")

    temp = temp.sort_values(["_date", "_pos"], kind="mergesort").reset_index(drop=True)

    date_counts = temp.groupby("_date", sort=True).size()
    unique_dates = list(date_counts.index)

    all_splits = []
    fold_rows = []

    start_date_pos = 0
    fold_num = 1

    while start_date_pos < len(unique_dates):
        first_test_date = unique_dates[start_date_pos]

        past_mask = temp["_date"] < first_test_date
        past_rows = temp.loc[past_mask]

        past_seasons = past_rows[season_col].dropna().drop_duplicates().tolist()
        if len(past_seasons) < train_seasons:
            start_date_pos += 1
            continue

        selected_train_seasons = past_seasons[-train_seasons:]

        train_mask = past_mask & temp[season_col].isin(selected_train_seasons)
        train_idx = temp.loc[train_mask, "_pos"].to_numpy()

        if len(train_idx) < min_train_games:
            start_date_pos += 1
            continue

        test_end_pos, _ = _advance_until_n_games(
            date_counts=date_counts,
            start_pos=start_date_pos,
            target_games=test_games,
        )

        if test_end_pos <= start_date_pos:
            break

        test_dates = unique_dates[start_date_pos:test_end_pos]
        test_mask = temp["_date"].isin(test_dates)
        test_idx = temp.loc[test_mask, "_pos"].to_numpy()

        if len(test_idx) == 0:
            break

        all_splits.append((train_idx, test_idx))

        fold_rows.append(
            {
                "fold": fold_num,
                "train_n_games": int(len(train_idx)),
                "test_n_games": int(len(test_idx)),
                "train_start_date": temp.loc[train_mask, "_date"].min(),
                "train_end_date": temp.loc[train_mask, "_date"].max(),
                "test_start_date": min(test_dates),
                "test_end_date": max(test_dates),
                "train_seasons": list(selected_train_seasons),
            }
        )

        next_start_pos, _ = _advance_until_n_games(
            date_counts=date_counts,
            start_pos=start_date_pos,
            target_games=step_games,
        )
        start_date_pos = max(start_date_pos + 1, next_start_pos)
        fold_num += 1

    if len(all_splits) == 0:
        raise ValueError("No valid folds were created.")

    fold_info = pd.DataFrame(fold_rows)

    # Randomly subsample folds if requested
    if max_folds is not None and max_folds < len(all_splits):
        rng = np.random.default_rng(random_state)
        selected_idx = rng.choice(len(all_splits), size=max_folds, replace=False)

        # Optional: sort selected folds by original order to preserve chronology
        selected_idx = np.sort(selected_idx)

        splits = [all_splits[i] for i in selected_idx]
        fold_info = fold_info.iloc[selected_idx].reset_index(drop=True)
    else:
        splits = all_splits
        fold_info = fold_info.reset_index(drop=True)

    # Re-label fold numbers after sampling
    fold_info["fold"] = np.arange(1, len(fold_info) + 1)

    if verbose >= 1:
        print(f"Created {len(splits)} walk-forward folds")
        if not fold_info.empty:
            print(fold_info.to_string(index=False))

    return splits, fold_info

In [10]:
def split_latest_dates_holdout(
    df: pd.DataFrame,
    date_col: str = "GAME_DATE",
    test_size: float = 0.15,
):
    temp = df.copy()
    temp[date_col] = pd.to_datetime(temp[date_col]).dt.normalize()

    date_counts = temp.groupby(date_col).size().sort_index()
    target_n_games = int(np.ceil(len(temp) * test_size))

    selected_test_dates = []
    running_n = 0

    for dt, n in reversed(list(date_counts.items())):
        selected_test_dates.append(dt)
        running_n += int(n)
        if running_n >= target_n_games:
            break

    selected_test_dates = set(selected_test_dates)
    test_mask = temp[date_col].isin(selected_test_dates)

    df_dev = df.loc[~test_mask].copy().reset_index(drop=True)
    df_test = df.loc[test_mask].copy().reset_index(drop=True)

    return df_dev, df_test

In [52]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.05,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 4413
Final test set size: 241
Final test date range: 2026-01-30 00:00:00 -> 2026-03-09 00:00:00


In [53]:
TARGET_COL = "LINE_ERROR"

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "LINE_ERROR",
    # "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

X_dev shape: (4413, 2084)
X_test_final shape: (241, 2084)


In [26]:
def over_under_betting_accuracy(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    true_sign = np.sign(y_true)
    pred_sign = np.sign(y_pred)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return 0.0

    return float(np.mean(true_sign[valid] == pred_sign[valid]))



ou_scorer = make_scorer(over_under_betting_accuracy, greater_is_better=True)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc == "OU_Betting_Accuracy":
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()

In [41]:
splits, fold_info = make_walk_forward_last_n_seasons_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    train_seasons=3,
    test_games=15,      # a few matches ahead
    step_games=30,      # next fold also moves by ~30 games
    min_train_games=450,
    max_folds = 10,
    random_state=16,
    verbose=1,
)

Created 10 walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date      train_seasons
    1           2346            15       2021-10-29     2023-11-10      2023-11-15    2023-12-12 [2021, 2022, 2023]
    2           2484            15       2021-10-29     2024-05-03      2024-05-04    2024-05-12 [2021, 2022, 2023]
    3           1701            19       2022-10-24     2024-12-22      2024-12-23    2024-12-25 [2022, 2023, 2024]
    4           1773            15       2022-10-24     2025-01-01      2025-01-02    2025-01-03 [2022, 2023, 2024]
    5           1867            16       2022-10-24     2025-01-14      2025-01-15    2025-01-16 [2022, 2023, 2024]
    6           1935            17       2022-10-24     2025-01-23      2025-01-24    2025-01-25 [2022, 2023, 2024]
    7           1965            15       2022-10-24     2025-01-27      2025-01-28    2025-01-29 [2022, 2023, 2024]
    8           2168            15       2

In [42]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MSE: 301.09483
Validation MSE: 313.64977

Train RMSE: 17.35187
Validation RMSE: 17.60917

Train MAE: 13.72522
Validation MAE: 14.41577

Train R2: 0.00000
Validation R2: -0.03120

Train OU_Betting_Accuracy: 51.53%
Validation OU_Betting_Accuracy: 46.11%



In [43]:
y_pred_zero = np.zeros(len(y_test_final))

mse = mean_squared_error(y_test_final, y_pred_zero)
rmse = root_mean_squared_error(y_test_final, y_pred_zero)
mae = mean_absolute_error(y_test_final, y_pred_zero)
ou_acc = over_under_betting_accuracy(y_test_final, y_pred_zero)

print("Zero-error baseline on final test")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Zero-error baseline on final test
MSE: 302.04730
RMSE: 17.37951
MAE: 13.77312
OU_Betting_Accuracy: 0.00%


In [44]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MSE: 88.59993
Validation MSE: 14618193.60157

Train RMSE: 9.32799
Validation RMSE: 1237.62430

Train MAE: 7.26033
Validation MAE: 639.35786

Train R2: 0.70502
Validation R2: -50188.53404

Train OU_Betting_Accuracy: 81.39%
Validation OU_Betting_Accuracy: 52.41%



In [45]:
xgb_reg = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=350,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-2,          # choose your CPU count
    random_state=16,
)

cv_results = cross_validate(
    xgb_reg,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,          # leave at 1 because XGB already parallelizes internally
)

print("XGBoost")
print_metrics(cv_results)

XGBoost
Train MSE: 84.17228
Validation MSE: 328.77067

Train RMSE: 9.16030
Validation RMSE: 18.03433

Train MAE: 7.25162
Validation MAE: 14.82867

Train R2: 0.72014
Validation R2: -0.09052

Train OU_Betting_Accuracy: 89.44%
Validation OU_Betting_Accuracy: 54.10%



In [46]:
xgb_reg.fit(X_dev, y_dev)

y_pred_test_error = xgb_reg.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy(y_test_final, y_pred_test_error)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

Final test metrics
MSE: 320.02368
RMSE: 17.88921
MAE: 14.34549
OU_Betting_Accuracy: 48.63%


In [47]:
def error_sign_accuracy(y_true_error, y_pred_error) -> float:
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def evaluate_error_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_error: pd.Series,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    y_pred_error = np.asarray(model.predict(X_test), dtype=float)
    margin = np.abs(y_pred_error)

    rows = []
    n_total = len(y_test_error)
    y_true_error_np = y_test_error.to_numpy(dtype=float)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        acc = (
            np.nan
            if n == 0
            else error_sign_accuracy(y_true_error_np[mask], y_pred_error[mask])
        )

        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_error


results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,703,100.0%,48.63%
1,1,570,81.1%,47.33%
2,2,409,58.2%,46.15%
3,3,295,42.0%,47.40%
4,4,200,28.4%,43.15%
5,5,139,19.8%,42.34%
6,6,70,10.0%,40.58%
7,7,42,6.0%,45.24%
8,8,26,3.7%,46.15%
9,9,11,1.6%,45.45%


In [48]:
import json
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd

from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from xgboost import XGBRegressor


# =========================================================
# Betting metrics
# =========================================================
def over_under_betting_accuracy(y_true_error, y_pred_error) -> float:
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return 0.0

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def ou_accuracy_with_threshold_errors(
    true_error: np.ndarray,
    pred_error: np.ndarray,
    threshold_abs: float = 0.0,
    min_coverage: float = 0.25,
):
    """
    Only bet when |predicted_error| > threshold_abs.
    Penalize solutions with too little coverage.
    """
    true_error = np.asarray(true_error, dtype=float)
    pred_error = np.asarray(pred_error, dtype=float)

    margin = np.abs(pred_error)
    mask = margin > threshold_abs
    coverage = float(np.mean(mask)) if len(mask) else 0.0

    if coverage < min_coverage or mask.sum() == 0:
        return 0.0, coverage

    score = over_under_betting_accuracy(true_error[mask], pred_error[mask])

    if np.isnan(score):
        return 0.0, coverage

    return float(score), coverage


def error_sign_accuracy(y_true_error, y_pred_error) -> float:
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def evaluate_error_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_error: pd.Series,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    y_pred_error = np.asarray(model.predict(X_test), dtype=float)
    margin = np.abs(y_pred_error)

    rows = []
    n_total = len(y_test_error)
    y_true_error_np = y_test_error.to_numpy(dtype=float)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        acc = (
            np.nan
            if n == 0
            else error_sign_accuracy(y_true_error_np[mask], y_pred_error[mask])
        )

        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_error


# =========================================================
# Helper to use best iteration after early stopping
# =========================================================
def _predict_best(model, X):
    if getattr(model, "best_iteration", None) is not None:
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            ntree_limit = getattr(model, "best_ntree_limit", None)
            if ntree_limit is not None:
                return model.predict(X, ntree_limit=ntree_limit)

    return model.predict(X)


# =========================================================
# Optuna objective using YOUR walk-forward splits
# =========================================================
def objective(trial, X, y, splits):
    threshold_abs = trial.suggest_float("threshold_abs", 0.0, 10.0, step=0.5)
    min_coverage = 0.25

    params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.80),
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.30, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 225, 550, step=25),
        "eval_metric": "rmse",
        "early_stopping_rounds": 200,
        "random_state": 16,
        "n_jobs": -1,
    }

    fold_scores = []
    fold_coverages = []

    for fold, (tr_idx, va_idx) in enumerate(splits, start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y.iloc[tr_idx].to_numpy(dtype=float)
        y_va = y.iloc[va_idx].to_numpy(dtype=float)

        model = XGBRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        y_pred_error = _predict_best(model, X_va)

        score, coverage = ou_accuracy_with_threshold_errors(
            true_error=y_va,
            pred_error=y_pred_error,
            threshold_abs=threshold_abs,
            min_coverage=min_coverage,
        )

        fold_scores.append(score)
        fold_coverages.append(coverage)

        trial.report(float(np.mean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("mean_coverage", float(np.mean(fold_coverages)))
    return float(np.mean(fold_scores))


# =========================================================
# Run Optuna on development set
# =========================================================
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=16),
    pruner=MedianPruner(n_warmup_steps=2),
)

study.optimize(
    lambda t: objective(t, X_dev, y_dev, splits),
    timeout=2.5 * 3600,   # or reduce if needed
    n_jobs=1,
)

print("Best value (CV OU success):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("Mean coverage:", study.best_trial.user_attrs.get("mean_coverage"))

[I 2026-03-10 17:18:11,804] A new study created in memory with name: no-name-1c6b1ef0-4b06-4cf7-a05b-3c69fb749d72
[I 2026-03-10 17:22:56,821] Trial 0 finished with value: 0.30777777777777776 and parameters: {'threshold_abs': 2.0, 'max_depth': 4, 'min_child_weight': 10.728161890750807, 'gamma': 0.22800975066403162, 'subsample': 0.6164373012086636, 'colsample_bytree': 0.5669242825073867, 'learning_rate': 0.05082275623350138, 'reg_alpha': 0.004517786465395959, 'reg_lambda': 0.01706650116663653, 'n_estimators': 550}. Best is trial 0 with value: 0.30777777777777776.
[I 2026-03-10 17:26:13,888] Trial 1 finished with value: 0.0 and parameters: {'threshold_abs': 5.5, 'max_depth': 2, 'min_child_weight': 13.615793124046794, 'gamma': 0.7922608676158321, 'subsample': 0.550168784046414, 'colsample_bytree': 0.5880461768306593, 'learning_rate': 0.05316051828678, 'reg_alpha': 0.07195423366266383, 'reg_lambda': 0.05127746977873438, 'n_estimators': 375}. Best is trial 0 with value: 0.30777777777777776.


Best value (CV OU success): 0.5822467320261437
Best params:
threshold_abs: 0.0
max_depth: 6
min_child_weight: 6.541512273488518
gamma: 4.635807733550123
subsample: 0.6240976810296368
colsample_bytree: 0.6901701866576757
learning_rate: 0.06514698176528576
reg_alpha: 1.1332400508119107
reg_lambda: 0.03585962503790806
n_estimators: 225
Mean coverage: 1.0


In [49]:
def train_final_model_time_aware(
    X_all: pd.DataFrame,
    y_all: pd.Series,
    study,
    *,
    use_early_stopping: bool = True,
    val_frac: float = 0.10,
    random_state: int = 16,
):
    best_params = study.best_params.copy()
    best_threshold = float(best_params.pop("threshold_abs"))

    final_params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "random_state": random_state,
        "n_jobs": -1,
        "eval_metric": "rmse",
        **best_params,
    }

    model = XGBRegressor(**final_params)

    if not use_early_stopping:
        model.fit(X_all, y_all, verbose=False)
        return model, best_threshold

    n = len(X_all)
    n_val = max(1, int(round(val_frac * n)))

    X_tr = X_all.iloc[:-n_val]
    y_tr = y_all.iloc[:-n_val]
    X_va = X_all.iloc[-n_val:]
    y_va = y_all.iloc[-n_val:]

    model.set_params(early_stopping_rounds=200)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )

    return model, best_threshold


final_model, best_threshold = train_final_model_time_aware(
    X_all=X_dev,
    y_all=y_dev,
    study=study,
    use_early_stopping=True,
    val_frac=0.10,
    random_state=16,
)

In [54]:

y_pred_test_error = _predict_best(final_model, X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy(y_test_final, y_pred_test_error)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")

results_df_final, _ = evaluate_error_thresholds(
    model=final_model,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df_final.style.format(
        {
            "pct_of_test": "{:.1%}",
            "directional_accuracy": "{:.2%}",
        }
    )
)

print(f"Optuna-selected threshold_abs: {best_threshold}")

Final test metrics
MSE: 299.77685
RMSE: 17.31407
MAE: 13.89033
OU_Betting_Accuracy: 50.21%


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,241,100.0%,50.21%
1,1,49,20.3%,51.06%
2,2,6,2.5%,33.33%
3,3,0,0.0%,nan%
4,4,0,0.0%,nan%
5,5,0,0.0%,nan%
6,6,0,0.0%,nan%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


Optuna-selected threshold_abs: 0.0


In [51]:
def save_model_bundle(
    model: XGBRegressor,
    threshold_abs: float,
    feature_names: list[str],
    out_dir: str | Path,
    name: str = "xgb_ou_model",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"{name}.joblib"
    meta_path = out_dir / f"{name}.meta.json"

    joblib.dump(model, model_path)

    metadata = {
        "threshold_abs": float(threshold_abs),
        "feature_names": list(feature_names),
        "xgboost_version_note": "Saved via joblib XGBRegressor wrapper",
    }
    meta_path.write_text(json.dumps(metadata, indent=2))

    return model_path, meta_path


def load_model_bundle(model_path: str | Path, meta_path: str | Path):
    model = joblib.load(model_path)
    metadata = json.loads(Path(meta_path).read_text())
    return model, metadata


model_path, meta_path = save_model_bundle(
    model=final_model,
    threshold_abs=best_threshold,
    feature_names=list(X_dev.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/error_line/",
    name="drop_na_cols_50_recent_data_xgb_line_error_10_03_26",
)

print("Saved:", model_path)
print("Saved:", meta_path)

Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/error_line/drop_na_cols_50_recent_data_xgb_line_error_10_03_26.joblib
Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/error_line/drop_na_cols_50_recent_data_xgb_line_error_10_03_26.meta.json
